# YOLOv11 — Belarus Road Signs
## Полный пайплайн: RTSD (Kaggle) → ремаппинг → обучение

### 1. Установка зависимостей

In [1]:
!pip install -q ultralytics kaggle

import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.2 MB/s eta 0:00:00
CUDA: True
GPU: Tesla T4


### 2. Kaggle API + скачивание датасета

In [ ]:
import os, json

# Вставь свои данные
KAGGLE_USERNAME = 'astelia'  # <-- поменяй
KAGGLE_KEY      = 'KGAT_fd110cc0bb1391fe5f50f81dd3912dad'       # <-- поменяй

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle.json записан ✓')

kaggle.json записан ✓


In [ ]:
# Скачиваем датасет (~18 GB, займёт 10-15 минут)
!kaggle datasets download -d watchman/rtsd-dataset -p /content/ --unzip

Dataset URL: https://www.kaggle.com/datasets/watchman/rtsd-dataset
License(s): unknown
100% 17.1G/17.1G [02:28<00:00, 124MB/s]



In [ ]:
!ls /content/rtsd-frames/

rtsd-frames


### 3. Конвертация JSON аннотаций → YOLO формат + ремаппинг классов

In [ ]:
import json
import shutil
from pathlib import Path
from PIL import Image
from tqdm import tqdm

# ── Маппинг: RTSD имя класса → новый sequential ID (0-102) ──────────
# Ключи — имена из JSON аннотаций (напр. '2_1', '3_24_n40')
# Значения — новый BY class ID
RTSD_NAME_TO_NEW_ID = {
    # WARNING
    '1_1':    82,  # 1.1  Ж/д переезд со шлагбаумом
    '1_2':    55,  # 1.2  Ж/д переезд без шлагбаума
    '1_5':    88,  # 1.5  Пересечение с трамвайной линией
    '1_7':    76,  # 1.7  Пересечение с круговым движением
    '1_8':    37,  # 1.8  Светофорное регулирование
    '1_10':  100,  # 1.10 Выезд на набережную
    '1_11':   42,  # 1.11.1 Опасный поворот направо
    '1_11_1': 25,  # 1.11.2 Опасный поворот налево
    '1_12':   45,  # 1.12.1 Опасные повороты (первый направо)
    '1_12_2': 43,  # 1.12.2 Опасные повороты (первый налево)
    '1_13':   71,  # 1.13 Крутой спуск
    '1_14':   72,  # 1.14 Крутой подъём
    '1_15':   18,  # 1.15 Скользкая дорога
    '1_16':   24,  # 1.16.2-4 Неровная дорога
    '1_17':    2,  # 1.16.1 Искусственная неровность
    '1_18':   78,  # 1.17 Выброс щебня
    '1_19':   22,  # 1.32 Опасная обочина
    '1_20':   44,  # 1.18.1 Сужение дороги с обеих сторон
    '1_21':   69,  # 1.19 Двустороннее движение
    '1_22':   34,  # 1.20 Впереди пешеходный переход
    '1_23':    1,  # 1.21 Дети
    '1_25':   11,  # 1.23 Дорожные работы
    '1_26':   98,  # 1.24 Перегон скота
    '1_27':   35,  # 1.25 Дикие животные
    '1_30':   92,  # 1.28 Низколетящие самолёты
    '1_33':   16,  # 1.30 Прочие опасности
    # PRIORITY
    '2_1':     0,  # 2.1  Главная дорога
    '2_2':     8,  # 2.2  Конец главной дороги
    '2_3':    39,  # 2.3.1 Пересечение со второстепенной
    '2_3_2':  36,  # 2.3.2 Примыкание справа
    '2_3_3':  40,  # 2.3.3 Примыкание слева
    '2_3_4':  73,  # 2.3.4 Пересечение равнозначных
    '2_4':     9,  # 2.4  Уступить дорогу
    '2_5':    47,  # 2.5  Стоп
    '2_6':    74,  # 2.6  Преимущество встречного
    '2_7':    79,  # 2.7  Преимущество перед встречным
    # PROHIBITORY
    '3_1':    48,  # 3.1  Въезд запрещён
    '3_2':    50,  # 3.2  Движение запрещено
    '3_4':    12,  # 3.4  Грузовые запрещены
    '3_6':    97,  # 3.6  Тракторы запрещены
    '3_10':   60,  # 3.10 Пешеходы запрещены
    '3_11':   84,  # 3.11 Ограничение массы
    '3_12':   94,  # 3.12 Ограничение на ось
    '3_13':   38,  # 3.13 Ограничение высоты
    '3_14':   54,  # 3.14 Ограничение ширины
    '3_16':   91,  # 3.16 Минимальная дистанция
    '3_18':   28,  # 3.18.1 Поворот направо запрещён
    '3_18_2': 75,  # 3.18.2 Поворот налево запрещён
    '3_19':   77,  # 3.19 Разворот запрещён
    '3_20':   49,  # 3.20 Обгон запрещён
    '3_21':   70,  # 3.21 Конец зоны запрета обгона
    '3_24':    3,  # 3.24 Ограничение скорости (все варианты)
    '3_24_n20': 3,
    '3_24_n40': 3,
    '3_24_n60': 3,
    '3_24_n80': 3,
    '3_24_n90': 3,
    '3_24_n100': 3,
    '3_24_n110': 3,
    '3_24_n120': 3,
    '3_25':    6,  # 3.25 Конец ограничения скорости
    '3_27':   17,  # 3.27 Остановка запрещена
    '3_28':   63,  # 3.28 Стоянка запрещена
    '3_29':   89,  # 3.29 Стоянка по нечётным
    '3_30':   85,  # 3.30 Стоянка по чётным
    '3_31':   67,  # 3.31 Конец всех ограничений
    '3_32':   46,  # 3.32 Опасные грузы запрещены
    '3_33':   95,  # 3.33 Зона ограничения скорости
    # MANDATORY
    '4_1_1':  15,  # 4.1.1 Прямо
    '4_1_2':  32,  # 4.1.2 Направо
    '4_1_3':  64,  # 4.1.3 Налево
    '4_1_4':  56,  # 4.1.4 Прямо или направо
    '4_1_5':  59,  # 4.1.5 Прямо или налево
    '4_1_6':  13,  # 4.1.6 Направо или налево
    '4_2_1':  10,  # 4.2.1 Объезд справа
    '4_2_2':  61,  # 4.2.2 Объезд слева
    '4_2_3':  14,  # 4.2.3 Объезд справа или слева
    '4_3':    58,  # 4.3  Круговое движение
    '4_5':    83,  # 4.6.1 Велосипедная дорожка
    # OTHER / INFO
    '5_3':    66,  # 5.3  Дорога для автомобилей
    '5_4':    65,  # 5.4  Конец дороги для автомобилей
    '5_5':    30,  # 5.5  Одностороннее движение
    '5_6':    29,  # 5.6  Конец одностороннего
    '5_7_1':  86,  # 5.7.1 Выезд на одностороннюю
    '5_7_2':  87,  # 5.7.2 Выезд на одностороннюю
    '5_8':    96,  # 5.35 Реверсивное движение
    '5_11':   93,  # 5.10.1 Дорога с полосой для маршрутных
    '5_12':   90,  # 5.10.4 Конец дороги с полосой
    '5_14':   80,  # 5.9.1 Полоса для маршрутных
    '5_16':    5,  # 5.16 Пешеходный переход
    '5_17':   99,  # 5.13.1 Остановочный пункт трамвая
    '5_18':   52,  # 5.14.2 Стоянка такси
    '5_19_1':  4,  # 5.16 Пешеходный переход (дубль)
    '5_21':   81,  # 5.38 Жилая зона
    '5_22':   51,  # 5.39 Конец жилой зоны
    # SERVICE
    '6_2':    68,  # 5.18.1 Рекомендуемая скорость
    '6_3_1':  19,  # 5.11.1 Место для разворота
    '6_4':    23,  # 5.15 Место стоянки
    '6_6':    26,  # 5.17.1-2 Подземный переход
    '6_7':    20,  # 5.17.3-4 Надземный переход
    '6_16':    7,  # 6.16 BelToll
    # ADDITIONAL
    '7_1':    62,  # 6.1 Первая помощь
    '7_2':    27,  # 6.2 Больница
    '7_3':    21,  # 6.3 АЗС
    '7_4':    31,  # 6.4 Техобслуживание
    '7_5':    53,  # 6.5 Мойка
    '7_6':    57,  # 6.6 Телефон
    '7_7':    41,  # 6.7 Питание
    '7_11':   33,  # 6.11 Место отдыха
    '7_14':  102,  # 6.14 Пункт контроля
    '7_18':  101,  # 6.13 Туалет
}



print(f'Классов в маппинге: {len(RTSD_NAME_TO_NEW_ID)}')
print(f'Уникальных BY ID: {len(set(RTSD_NAME_TO_NEW_ID.values()))}')

Классов в маппинге: 111
Уникальных BY ID: 103


In [ ]:
import json
import shutil
from pathlib import Path
from tqdm import tqdm

def process_split_coco(anno_json_path, frames_base_dir, out_images_dir, out_labels_dir):
    Path(out_images_dir).mkdir(parents=True, exist_ok=True)
    Path(out_labels_dir).mkdir(parents=True, exist_ok=True)

    with open(anno_json_path) as f:
        anno = json.load(f)

    # Строим словари для быстрого доступа
    # image_id -> {width, height, file_name}
    id_to_image = {img['id']: img for img in anno['images']}

    # category_id -> name (напр. '2_1', '3_24_n40')
    id_to_catname = {cat['id']: cat['name'] for cat in anno['categories']}

    # image_id -> [annotations]
    from collections import defaultdict
    img_to_annos = defaultdict(list)
    for a in anno['annotations']:
        img_to_annos[a['image_id']].append(a)

    skipped_classes = set()
    total_boxes = written_boxes = skipped_boxes = 0
    copied_images = 0

    for img_id, img_info in tqdm(id_to_image.items(), desc=Path(anno_json_path).stem):
        # file_name в JSON: 'rtsd-frames/autosave...jpg'
        # реальный путь: /content/rtsd-frames/rtsd-frames/autosave...jpg
        filename = Path(img_info['file_name']).name  # берём только имя файла
        img_src = Path(frames_base_dir) / filename
        if not img_src.exists():
            continue

        img_w = img_info['width']
        img_h = img_info['height']

        yolo_lines = []
        for a in img_to_annos[img_id]:
            total_boxes += 1
            cat_name = id_to_catname.get(a['category_id'], '')
            new_id = RTSD_NAME_TO_NEW_ID.get(cat_name)
            if new_id is None:
                skipped_classes.add(cat_name)
                skipped_boxes += 1
                continue

            # COCO bbox: [x_min, y_min, width, height]
            x, y, w, h = a['bbox']
            xc = (x + w / 2) / img_w
            yc = (y + h / 2) / img_h
            wn = w / img_w
            hn = h / img_h
            xc = max(0.0, min(1.0, xc))
            yc = max(0.0, min(1.0, yc))
            wn = max(0.0, min(1.0, wn))
            hn = max(0.0, min(1.0, hn))
            yolo_lines.append(f'{new_id} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}')
            written_boxes += 1

        dst_img = Path(out_images_dir) / filename
        shutil.copy2(img_src, dst_img)
        copied_images += 1

        label_name = Path(filename).stem + '.txt'
        with open(Path(out_labels_dir) / label_name, 'w') as f:
            f.write('\n'.join(yolo_lines) + ('\n' if yolo_lines else ''))

    print(f'  Изображений: {copied_images}')
    print(f'  Bbox: записано={written_boxes}, пропущено={skipped_boxes} / всего={total_boxes}')
    if skipped_classes:
        print(f'  Пропущенные классы: {sorted(skipped_classes)}')


FRAMES_DIR = '/content/rtsd-frames/rtsd-frames'
OUT_DIR    = '/content/belarus_yolo'

process_split_coco('/content/train_anno.json', FRAMES_DIR,
                   f'{OUT_DIR}/train/images', f'{OUT_DIR}/train/labels')

process_split_coco('/content/val_anno.json', FRAMES_DIR,
                   f'{OUT_DIR}/val/images', f'{OUT_DIR}/val/labels')

print('\nКонвертация завершена ✓')


train_anno: 100%|██████████| 54188/54188 [01:29<00:00, 607.04it/s] 


  Изображений: 54188
  Bbox: записано=78627, пропущено=16865 / всего=95492
  Пропущенные классы: ['1_20_2', '1_20_3', '1_31', '1_6', '2_3_5', '2_3_6', '3_4_1', '4_1_2_1', '4_1_2_2', '4_8_2', '4_8_3', '5_15_1', '5_15_2', '5_15_2_2', '5_15_3', '5_15_5', '5_15_7', '5_20', '6_15_1', '6_15_2', '6_15_3', '6_8_1', '6_8_2', '6_8_3', '7_12', '7_15', '8_13', '8_13_1', '8_14', '8_15', '8_16', '8_17', '8_18', '8_1_1', '8_1_3', '8_1_4', '8_23', '8_2_1', '8_2_2', '8_2_3', '8_2_4', '8_3_1', '8_3_2', '8_3_3', '8_4_1', '8_4_3', '8_4_4', '8_5_2', '8_5_4', '8_6_2', '8_6_4', '8_8']


val_anno: 100%|██████████| 5000/5000 [00:12<00:00, 415.79it/s]

  Изображений: 5000
  Bbox: записано=7303, пропущено=1563 / всего=8866
  Пропущенные классы: ['1_20_2', '1_20_3', '1_31', '2_3_6', '3_4_1', '4_1_2_1', '4_1_2_2', '4_8_2', '5_15_1', '5_15_2', '5_15_2_2', '5_15_3', '5_15_5', '5_15_7', '5_20', '6_15_1', '6_15_2', '6_8_1', '7_12', '7_15', '8_13', '8_13_1', '8_15', '8_17', '8_18', '8_1_1', '8_1_3', '8_1_4', '8_2_1', '8_2_2', '8_2_3', '8_2_4', '8_3_1', '8_3_2', '8_4_1', '8_4_3', '8_4_4', '8_5_2', '8_5_4', '8_6_2', '8_6_4', '8_8']

Конвертация завершена ✓


### 4. Проверка датасета

In [ ]:
from pathlib import Path
from collections import Counter

NEW_ID_TO_BY = {
    0:'2.1', 1:'1.21', 2:'1.16.1', 3:'3.24', 4:'5.16', 5:'5.16',
    6:'3.25', 7:'6.16', 8:'2.2', 9:'2.4', 10:'4.2.1', 11:'1.23',
    12:'3.4', 13:'4.1.6', 14:'4.2.3', 15:'4.1.1', 16:'1.30', 17:'3.27',
    18:'1.15', 19:'5.11.1', 20:'5.17.3-4', 21:'6.3', 22:'1.32', 23:'5.15',
    24:'1.16.2-4', 25:'1.11.2', 26:'5.17.1-2', 27:'6.2', 28:'3.18.1',
    29:'5.6', 30:'5.5', 31:'6.4', 32:'4.1.2', 33:'6.11', 34:'1.20',
    35:'1.25', 36:'2.3.2', 37:'1.8', 38:'3.13', 39:'2.3.1', 40:'2.3.3',
    41:'6.7', 42:'1.11.1', 43:'1.12.2', 44:'1.18.1', 45:'1.12.1',
    46:'3.32', 47:'2.5', 48:'3.1', 49:'3.20', 50:'3.2', 51:'5.39',
    52:'5.14.2', 53:'6.5', 54:'3.14', 55:'1.2', 56:'4.1.4', 57:'6.6',
    58:'4.3', 59:'4.1.5', 60:'3.10', 61:'4.2.2', 62:'6.1', 63:'3.28',
    64:'4.1.3', 65:'5.4', 66:'5.3', 67:'3.31', 68:'5.18.1', 69:'1.19',
    70:'3.21', 71:'1.13', 72:'1.14', 73:'2.3.4', 74:'2.6', 75:'3.18.2',
    76:'1.7', 77:'3.19', 78:'1.17', 79:'2.7', 80:'5.9.1', 81:'5.38',
    82:'1.1', 83:'4.6.1', 84:'3.11', 85:'3.30', 86:'5.7.1', 87:'5.7.2',
    88:'1.5', 89:'3.29', 90:'5.10.4', 91:'3.16', 92:'1.28', 93:'5.10.1',
    94:'3.12', 95:'3.33', 96:'5.35', 97:'3.6', 98:'1.24', 99:'5.13.1',
    100:'1.10', 101:'6.13', 102:'6.14',
}

counter = Counter()
label_files = list(Path('/content/belarus_yolo/train/labels').rglob('*.txt'))
for f in label_files:
    with open(f) as fp:
        for line in fp:
            parts = line.strip().split()
            if parts:
                counter[int(parts[0])] += 1

all_ids = set(counter.keys())
print(f'Train images:  {len(label_files)}')
print(f'Классов с примерами: {len(all_ids)} / 103')
print(f'ID в диапазоне 0-102: {max(all_ids) <= 102 and min(all_ids) >= 0}')
print(f'\nТоп-15 классов:')
for new_id, cnt in counter.most_common(15):
    print(f'  {NEW_ID_TO_BY[new_id]:12s} (id={new_id:3d})  {cnt} bbox')

missing = set(range(103)) - all_ids
print(f'\nОтсутствуют ({len(missing)} классов): {sorted(missing)}')

Train images:  54188
Классов с примерами: 103 / 103
ID в диапазоне 0-102: True

Топ-15 классов:
  5.16         (id=  4)  22147 bbox
  2.1          (id=  0)  10027 bbox
  5.16         (id=  5)  4727 bbox
  3.24         (id=  3)  3761 bbox
  2.4          (id=  9)  3755 bbox
  3.27         (id= 17)  3268 bbox
  1.21         (id=  1)  2558 bbox
  4.1.1        (id= 15)  2137 bbox
  3.20         (id= 49)  1661 bbox
  1.16.1       (id=  2)  1149 bbox
  4.2.3        (id= 14)  1114 bbox
  4.2.1        (id= 10)  1071 bbox
  4.1.4        (id= 56)  1063 bbox
  6.3          (id= 21)  1011 bbox
  5.15         (id= 23)  979 bbox

Отсутствуют (0 классов): []


In [ ]:
import os

# Смотрим что скачалось
!find /content -maxdepth 3 -type d | head -30
print("---")
# Ищем JSON аннотации
!find /content -name "*.json" | head -10
print("---")
# Ищем изображения
!find /content -name "*.jpg" | head -5


/content
/content/.config
/content/.config/configurations
/content/.config/logs
/content/.config/logs/2026.02.06
/content/rtsd-frames
/content/rtsd-frames/rtsd-frames
/content/belarus_yolo
/content/belarus_yolo/train
/content/belarus_yolo/train/images
/content/belarus_yolo/train/labels
/content/belarus_yolo/val
/content/belarus_yolo/val/images
/content/belarus_yolo/val/labels
/content/sample_data
---
/content/.config/.last_update_check.json
/content/train_anno.json
/content/train_anno_reduced.json
/content/val_anno.json
/content/label_map.json
/content/sample_data/anscombe.json
---
/content/rtsd-frames/rtsd-frames/autosave01_02_2012_09_13_33.jpg
/content/rtsd-frames/rtsd-frames/autosave01_02_2012_09_13_35.jpg
/content/rtsd-frames/rtsd-frames/autosave01_02_2012_09_13_36.jpg
/content/rtsd-frames/rtsd-frames/autosave01_02_2012_09_13_40.jpg
/content/rtsd-frames/rtsd-frames/autosave01_02_2012_09_13_41.jpg


In [ ]:
import json

with open('/content/train_anno.json') as f:
    anno = json.load(f)

print(f'Тип: {type(anno)}')
print(f'Ключи верхнего уровня: {list(anno.keys()) if isinstance(anno, dict) else "список"}')

# Если dict — смотрим первый ключ
if isinstance(anno, dict):
    first_key = list(anno.keys())[0]
    print(f'\nПервый ключ: {first_key}')
    print(f'Тип значения: {type(anno[first_key])}')
    print(f'Значение: {anno[first_key]}')


Тип: <class 'dict'>
Ключи верхнего уровня: ['images', 'annotations', 'categories']

Первый ключ: images
Тип значения: <class 'list'>
Значение: [{'id': 0, 'width': 1280, 'height': 720, 'file_name': 'rtsd-frames/autosave01_02_2012_09_13_33.jpg'}, {'id': 1, 'width': 1280, 'height': 720, 'file_name': 'rtsd-frames/autosave01_02_2012_09_13_34.jpg'}, {'id': 2, 'width': 1280, 'height': 720, 'file_name': 'rtsd-frames/autosave01_02_2012_09_13_35.jpg'}, {'id': 3, 'width': 1280, 'height': 720, 'file_name': 'rtsd-frames/autosave01_02_2012_09_13_36.jpg'}, {'id': 4, 'width': 1280, 'height': 720, 'file_name': 'rtsd-frames/autosave01_02_2012_09_13_37.jpg'}, {'id': 5, 'width': 1280, 'height': 720, 'file_name': 'rtsd-frames/autosave01_02_2012_09_13_38.jpg'}, {'id': 6, 'width': 1280, 'height': 720, 'file_name': 'rtsd-frames/autosave01_02_2012_09_13_39.jpg'}, {'id': 7, 'width': 1280, 'height': 720, 'file_name': 'rtsd-frames/autosave01_02_2012_09_13_42.jpg'}, {'id': 8, 'width': 1280, 'height': 720, 'file_na

### 5. Записываем data.yaml

In [ ]:
data_yaml = """train: /content/belarus_yolo/train/images
val: /content/belarus_yolo/val/images

nc: 103

names:
  - '2.1: Главная дорога'
  - '1.21: Дети'
  - '1.16.1: Искусственная неровность'
  - '3.24: Ограничение максимальной скорости'
  - '5.16: Пешеходный переход'
  - '5.16: Пешеходный переход'
  - '3.25: Конец зоны ограничения максимальной скорости'
  - '6.16: Пункт обслуживания системы BelToll'
  - '2.2: Конец главной дороги'
  - '2.4: Уступить дорогу'
  - '4.2.1: Объезд препятствия справа'
  - '1.23: Дорожные работы'
  - '3.4: Движение грузовых автомобилей запрещено'
  - '4.1.6: Движение направо или налево'
  - '4.2.3: Объезд препятствия справа или слева'
  - '4.1.1: Движение прямо'
  - '1.30: Прочие опасности'
  - '3.27: Остановка запрещена'
  - '1.15: Скользкая дорога'
  - '5.11.1: Место для разворота'
  - '5.17.3-4: Надземный пешеходный переход'
  - '6.3: Автозаправочная станция'
  - '1.32: Опасная обочина'
  - '5.15: Место стоянки'
  - '1.16.2-4: Неровная дорога'
  - '1.11.2: Опасный поворот (налево)'
  - '5.17.1-2: Подземный пешеходный переход'
  - '6.2: Больница'
  - '3.18.1: Поворот направо запрещён'
  - '5.6: Конец дороги с односторонним движением'
  - '5.5: Дорога с односторонним движением'
  - '6.4: Техническое обслуживание автомобилей'
  - '4.1.2: Движение направо'
  - '6.11: Место отдыха'
  - '1.20: Впереди пешеходный переход'
  - '1.25: Дикие животные'
  - '2.3.2: Примыкание второстепенной дороги (справа)'
  - '1.8: Светофорное регулирование'
  - '3.13: Ограничение высоты'
  - '2.3.1: Пересечение со второстепенной дорогой'
  - '2.3.3: Примыкание второстепенной дороги (слева)'
  - '6.7: Пункт питания'
  - '1.11.1: Опасный поворот (направо)'
  - '1.12.2: Опасные повороты (первый — налево)'
  - '1.18.1: Сужение дороги с обеих сторон'
  - '1.12.1: Опасные повороты (первый — направо)'
  - '3.32: Движение ТС с опасными грузами запрещено'
  - '2.5: Движение без остановки запрещено'
  - '3.1: Въезд запрещён'
  - '3.20: Обгон запрещён'
  - '3.2: Движение запрещено'
  - '5.39: Конец жилой зоны'
  - '5.14.2: Место стоянки такси'
  - '6.5: Мойка автомобилей'
  - '3.14: Ограничение ширины'
  - '1.2: Железнодорожный переезд без шлагбаума'
  - '4.1.4: Движение прямо или направо'
  - '6.6: Телефон'
  - '4.3: Круговое движение'
  - '4.1.5: Движение прямо или налево'
  - '3.10: Движение пешеходов запрещено'
  - '4.2.2: Объезд препятствия слева'
  - '6.1: Пункт первой медицинской помощи'
  - '3.28: Стоянка запрещена'
  - '4.1.3: Движение налево'
  - '5.4: Конец дороги для автомобилей'
  - '5.3: Дорога для автомобилей'
  - '3.31: Конец зоны всех ограничений'
  - '5.18.1: Рекомендуемая скорость'
  - '1.19: Двустороннее движение'
  - '3.21: Конец зоны запрещения обгона'
  - '1.13: Крутой спуск'
  - '1.14: Крутой подъём'
  - '2.3.4: Пересечение равнозначных дорог'
  - '2.6: Преимущество встречного движения'
  - '3.18.2: Поворот налево запрещён'
  - '1.7: Пересечение с круговым движением'
  - '3.19: Разворот запрещён'
  - '1.17: Выброс щебня'
  - '2.7: Преимущество перед встречным движением'
  - '5.9.1: Полоса для маршрутных тс'
  - '5.38: Жилая зона'
  - '1.1: Железнодорожный переезд со шлагбаумом'
  - '4.6.1: Велосипедная дорожка'
  - '3.11: Ограничение массы'
  - '3.30: Стоянка запрещена по чётным числам месяца'
  - '5.7.1: Выезд на дорогу с односторонним движением'
  - '5.7.2: Выезд на дорогу с односторонним движением'
  - '1.5: Пересечение с трамвайной линией'
  - '3.29: Стоянка запрещена по нечётным числам месяца'
  - '5.10.4: Конец дороги с полосой для маршрутных тс'
  - '3.16: Ограничение минимальной дистанции'
  - '1.28: Низколетящие самолеты'
  - '5.10.1: Дорога с полосой для маршрутных тс'
  - '3.12: Ограничение нагрузки на ось'
  - '3.33: Зона с ограничением максимальной скорости'
  - '5.35: Реверсивное движение'
  - '3.6: Движение тракторов запрещено'
  - '1.24: Перегон скота'
  - '5.13.1: Остановочный пункт трамвая'
  - '1.10: Выезд на набережную'
  - '6.13: Туалет'
  - '6.14: Пункт контроля автомобильных перевозок'
"""

with open('/content/belarus_yolo/data.yaml', 'w') as f:
    f.write(data_yaml)
print('data.yaml записан ✓')

data.yaml записан ✓


### 6. Монтируем Drive и копируем backbone

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Проверяем что last.pt существует
import os

last_pt = '/content/drive/MyDrive/Colab Notebooks/Raspberry/models/rtsd_v3_transfer/weights/last.pt'
best_pt = '/content/drive/MyDrive/Colab Notebooks/Raspberry/models/rtsd_v3_transfer/weights/best.pt'

print(f'last.pt существует: {os.path.exists(last_pt)}')
print(f'best.pt существует: {os.path.exists(best_pt)}')

if not os.path.exists(last_pt):
    print('\n⚠️  last.pt не найден — обучение ещё не запускалось или путь неверный')
    print('Проверь путь к папке с весами на Drive')

In [ ]:
# Копируем last.pt локально для быстрого доступа
import shutil

shutil.copy(last_pt, '/content/last.pt')
print('last.pt скопирован локально ✓')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy(
    '/content/drive/MyDrive/Raspberry/models/backbone_transfer.pt',
    '/content/backbone_transfer.pt'
)
print('backbone_transfer.pt скопирован ✓')

### 7. Обучение

In [ ]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
model = YOLO('/content/last.pt')

In [ ]:


# Загружаем last.pt — он уже содержит состояние обучения (эпоха, оптимайзер, lr)


model.train(
    data='/content/belarus_yolo/data.yaml',
    epochs=20,         # общее кол-во эпох — YOLO сам знает с какой продолжать
    imgsz=640,
    batch=32,
    device=0,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    freeze=22,
    amp=True,
    resume=True,       # продолжаем с последней эпохи
    project='/content/drive/MyDrive/Raspberry/models',
    name='rtsd_v3_transfer',
    exist_ok=True,
)
print('DONE')

Ultralytics 8.4.26 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/belarus_yolo/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=22, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=rtsd_v3_transfer, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=1